# 03: Evoformer

The Evoformer repeatedly exchanges information between two representations:

- `m`: residues across related protein sequences
- `z`: relationships between every pair of query residues

`z` guides attention in `m`, while evolutionary patterns in `m` update `z`. The extra MSA is processed first with memory-efficient global column attention.

**Read alongside:** `../src/af2_from_scratch/evoformer.py`

## Stage map

```text
                 pair information z
                         |
                         v
MSA m ---> row attention ---> column attention ---> transition
  |                                                  |
  +--------------- outer-product mean <--------------+
                         |
                         v
                  update pair grid z
                         |
                         v
       triangle multiplication and triangle attention
                         |
                         v
                  refined m and z
```

The extra-MSA blocks follow the same flow but replace ordinary column attention with global column attention, which averages the queries before attending across sequences.

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")  # package source lives one level up
torch.manual_seed(0)
plt.rcParams["figure.figsize"] = (8, 4)

In [ ]:
from af2_from_scratch import AF2Config
from af2_from_scratch.evoformer import (
    Evoformer,
    EvoformerBlock,
    OuterProductMean,
    TriMult,
)
from af2_from_scratch.feature_embedding import InputEmbedder
from af2_from_scratch.feature_extraction import msa_features, sample_batch

cfg = AF2Config()
features = msa_features("../examples/tautomerase/alignment.a3m")
batch = sample_batch(features, cfg.n_clu, cfg.n_ext, seed=0)
m, z, e = InputEmbedder(cfg)(batch)
print("m:", tuple(m.shape), "z:", tuple(z.shape), "e:", tuple(e.shape))

## 1. Exchange pair and MSA information with attention

**Row attention** compares residues within each related sequence. The pair representation `z` biases its attention scores, so current pair information guides sequence processing.

**Column attention** compares related sequences at one residue position. This allows the model to identify evolutionary patterns shared across sequences.

```text
                 residues
             r1  r2  r3  r4
sequence 1   ----------------> row attention
sequence 2   ---------------->
sequence 3   ---------------->
              |   |   |   |
              v   v   v   v
             column attention
```

Both attention operations use learned output gates. A gate independently controls how much of each attention output channel enters the residual update.

For the much larger extra MSA, global column attention averages the queries into one query per head. This changes attention memory from quadratic to linear in the number of extra sequences. Position-specific gates broadcast the global result back to every sequence.

In [ ]:
block = EvoformerBlock(cfg).eval()
with torch.no_grad():
    row_update = block.row(m, z)
    column_update = block.col(m)

print("row-attention update:", tuple(row_update.shape))
print("column-attention update:", tuple(column_update.shape))
print("row attention is gated:", block.row.mha.gate is not None)
print("column attention is gated:", block.col.mha.gate is not None)

## 2. Send MSA information to the pair grid

The outer-product mean sends information from `m` to `z`. For every residue pair `(i, j)`, it multiplies projected features from MSA columns `i` and `j`, then averages across sequences.

```text
MSA column i x MSA column j
             |
             v
     average across sequences
             |
             v
          update z[i, j]
```

These products are not a statistical covariance, but learned projections can use them to represent correlated evolutionary patterns.

In [ ]:
outer_product_mean = OuterProductMean(cfg.c_m, cfg.c_z)
pair_update = outer_product_mean(m)
print("outer-product update:", tuple(pair_update.shape))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(z[..., 0].detach(), cmap="RdBu")
axes[0].set_title("z before, channel 0")
axes[1].imshow((z + pair_update)[..., 0].detach(), cmap="RdBu")
axes[1].set_title("z after outer-product update")
plt.show()

## 3. Make residue-pair relationships consistent

Triangle updates refine the relationship between residues `i` and `j` using every possible third residue `k`. This allows the three pair relationships `(i, k)`, `(k, j)`, and `(i, j)` to become mutually consistent.

```text
        k
       /   z[i,k] z[k,j]
     /         i-------j
      z[i,j]
```

Outgoing and incoming triangle multiplication use different edge directions. Triangle attention then lets each pair select which third-residue relationships matter most.

In [ ]:
triangle_multiplication = TriMult(cfg.c_z, outgoing=True)
triangle_update = triangle_multiplication(z)
print("triangle update:", tuple(triangle_update.shape))

## 4. Run the full Evoformer

One main Evoformer block performs:

```text
MSA stack:
  row attention with pair bias
  column attention
  MSA transition
  outer-product mean: m -> z

Pair stack:
  outgoing triangle multiplication
  incoming triangle multiplication
  starting-node triangle attention
  ending-node triangle attention
  pair transition
```

Every operation contributes through a residual connection. AlphaFold's shared dropout uses one mask across a whole row or column instead of dropping every tensor element independently.

In [ ]:
evoformer = Evoformer(cfg).eval()
with torch.no_grad():
    m_out, z_out = evoformer(m, z, e)

print("after Evoformer: m", tuple(m_out.shape), "z", tuple(z_out.shape))
print(f"parameters: {sum(p.numel() for p in evoformer.parameters()) / 1e6:.2f}M")
print("pair representation changed:", not torch.allclose(z, z_out))
print(
    "extra MSA uses global column attention:",
    evoformer.extra[0].col.mha.global_attention,
)

## 5. Where are the parameters?

The following cell compares parameter counts across one block. It does not measure execution time or memory use, which also depend strongly on the number of sequences and residues.

In [ ]:
block = evoformer.blocks[0]
modules = [
    ("row attention", block.row),
    ("column attention", block.col),
    ("MSA transition", block.tm),
    ("outer-product mean", block.opm),
    ("triangle multiplication x2", torch.nn.ModuleList([block.tmo, block.tmi])),
    ("triangle attention x2", torch.nn.ModuleList([block.tas, block.tae])),
    ("pair transition", block.tp),
]
for name, module in modules:
    parameters = sum(parameter.numel() for parameter in module.parameters())
    print(f"{name:27s} {parameters / 1e3:7.1f}k")

**Recap:** The Evoformer alternates between sequence-alignment reasoning in `m` and residue-pair reasoning in `z`. Attention sends pair information into the MSA, while the outer-product mean sends evolutionary information back into the pair grid.

Next, `04_geometry.ipynb` introduces the coordinate tools used to turn learned representations into a 3D backbone trace.